# Paper classifier walkthrough

This notebook calls a locally served vLLM model through its OpenAI-compatible API. Change `model` to the name passed to, or exposed by, your vLLM server.

In [1]:
import sys
from pathlib import Path

# Support running Jupyter from either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "synth_extract").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from synth_extract.agents.classification import (
    ClassificationFailure,
    ClassificationResult,
    PaperClassifier,
)

## Configure the vLLM endpoint

In [7]:
base_url = "http://localhost:8000/v1"
api_key = "not-required"
model = "replace-with-your-vllm-served-model-name"

import os
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
model = "openai/gpt-5-mini"

classifier = PaperClassifier(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=60.0,
    max_tokens=10000,
)
classifier.llm_config()

{'model': 'openai/gpt-5-mini',
 'base_url': 'https://openrouter.ai/api/v1',
 'api_key_provided': True,
 'temperature': 0.0,
 'max_tokens': 10000,
 'timeout': 60.0,
 'max_retries': 0,
 'system_prompt_path': '/Users/kevinge/Work/Data Extraction/synth_extract/synth_extract/agents/classification/prompts/system_prompt.md',
 'user_template_path': '/Users/kevinge/Work/Data Extraction/synth_extract/synth_extract/agents/classification/prompts/user_template.md',
 'prompt_hash': 'b8a49f10d799b87013e4624202fe3665dbb9b9da83c9fb841e8ecc8deb32ed8b'}

## Inspect the request before calling the model

In [8]:
title = "Synthesis and thermal characterization of bio-based polyesters"
abstract = (
    "A series of bio-based polyesters was synthesized by melt "
    "polycondensation. Their molecular weights and glass-transition "
    "temperatures were measured using GPC and DSC."
)

print(classifier.render_prompt(title, abstract))

=== SYSTEM ===
You are a binary scientific-paper classification system.

Classify whether the supplied title and abstract describe an original experimental paper that is potentially relevant to a sample-level polymer synthesis and property dataset.

Return `true` when the paper appears to:
- study a polymeric or polymer-containing material experimentally, and
- synthesize, prepare, fabricate, process, modify, or characterize that material.

Return `false` when the paper is clearly:
- unrelated to polymers or polymer-containing materials,
- a review, editorial, correction, news item, or other non-original study,
- exclusively theoretical or computational with no experimentally studied material, or
- focused only on background or literature discussion.

Use only the supplied title and abstract. Do not assume facts that are not present.
When the evidence is limited or ambiguous, return `false`.

Return only the prescribed structured response.

=== USER ===
Classify the following paper.

T

## Classify the paper

In [9]:
result = classifier.classify(title=title, abstract=abstract)
result

ClassificationResult(label=True)

In [10]:
if isinstance(result, ClassificationResult):
    print(f"Classification label: {result.label}")
elif isinstance(result, ClassificationFailure):
    print(f"Classification failed ({result.error_type}): {result.message}")

Classification label: True


## Changing the classification prompt

Edit `synth_extract/agents/classification/prompts/system_prompt.md`, then run `classifier.reload_prompts()` before the next classification. You can also pass alternate prompt paths to `PaperClassifier`.